In [3]:
import pandas as pd
import numpy as np
import os

print("====================================================")
print("🎯 TASK 3: DATASET SELECTION & PROGRAMMATIC CLEANING")
print("====================================================\n")

# 1. Dataset Selection: Generate 5,000 realistic E-commerce transactions
np.random.seed(42)
raw_data = {
    'Transaction_ID': range(1001, 6001),
    'Date': pd.date_range(start='2026-01-01', periods=5000, freq='h'),
    'Customer_ID': [f"CUST-{np.random.randint(100, 999)}" for _ in range(5000)],
    'Product_Category': np.random.choice(['Electronics', 'Clothing', 'Home Decor', 'Beauty', 'Books'], size=5000, p=[0.3, 0.25, 0.2, 0.15, 0.1]),
    'Quantity': np.random.choice([1, 2, 3, 4, 5], size=5000, p=[0.5, 0.25, 0.15, 0.07, 0.03]),
    'Unit_Price': np.random.uniform(10.0, 500.0, size=5000).round(2),
    'Shipping_Cost': np.random.uniform(5.0, 50.0, size=5000).round(2),
    'Delivery_Status': np.random.choice(['Delivered', 'Shipped', 'Cancelled', 'Returned'], size=5000, p=[0.8, 0.1, 0.06, 0.04]),
    'Customer_Rating': np.random.choice([1, 2, 3, 4, 5], size=5000, p=[0.05, 0.05, 0.1, 0.3, 0.5]),
    'Region': np.random.choice(['North America', 'Europe', 'Asia', 'South America'], size=5000)
}
df = pd.DataFrame(raw_data)

# Inject synthetic errors (missing ratings & duplicate entries) to show cleaning steps
df.loc[df.sample(frac=0.02).index, 'Customer_Rating'] = np.nan
df = pd.concat([df, df.iloc[[0, 10, 20]]], ignore_index=True) 

# 2. Data Inspection
print(f"Data Inspection: Loaded {df.shape[0]} rows and {df.shape[1]} fields.")
print("\n--- Identified Initial Schema & Data Types ---")
print(df.dtypes)

# 3. Auditing for Duplicates and Missing Rows
print("\n--- Pre-Cleaning Quality Check ---")
print(f"Duplicate records caught: {df.duplicated().sum()}")
print("Missing entries detected per column:")
print(df.isnull().sum())

# 4. Data Cleaning Execution
df.drop_duplicates(inplace=True)
rating_median = df['Customer_Rating'].median()
df['Customer_Rating'].fillna(rating_median, inplace=True)

# 5. Feature Engineering: Create Total Sales Column
df['Total_Sales'] = (df['Quantity'] * df['Unit_Price']).round(2)

print("\n--- Post-Cleaning Validation ---")
print(f"Final normalized dataset dimension: {df.shape}")
print(f"Remaining null values across columns: {df.isnull().sum().sum()}")

# Create the dataset folder if it doesn't exist and save the file locally
os.makedirs('dataset', exist_ok=True)
df.to_csv('dataset/ecommerce_clean_data.csv', index=False)
print("\n🔥 Success: 'dataset/ecommerce_clean_data.csv' generated and saved locally!")


🎯 TASK 3: DATASET SELECTION & PROGRAMMATIC CLEANING

Data Inspection: Loaded 5003 rows and 10 fields.

--- Identified Initial Schema & Data Types ---
Transaction_ID               int64
Date                datetime64[us]
Customer_ID                    str
Product_Category               str
Quantity                     int64
Unit_Price                 float64
Shipping_Cost              float64
Delivery_Status                str
Customer_Rating            float64
Region                         str
dtype: object

--- Pre-Cleaning Quality Check ---
Duplicate records caught: 3
Missing entries detected per column:
Transaction_ID        0
Date                  0
Customer_ID           0
Product_Category      0
Quantity              0
Unit_Price            0
Shipping_Cost         0
Delivery_Status       0
Customer_Rating     100
Region                0
dtype: int64

--- Post-Cleaning Validation ---
Final normalized dataset dimension: (5000, 11)
Remaining null values across columns: 100

🔥 Succes

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7836\1462594233.py:43: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Customer_Rating'].fillna(rating_median, inplace=True)


In [5]:
import pandas as pd
import numpy as np

# Load our freshly cleaned dataset
df = pd.read_csv('dataset/ecommerce_clean_data.csv')

print("====================================================")
print("📊 TASK 4: DESCRIPTIVE STATISTICS & METRIC AUDITING")
print("====================================================\n")

# 1. Output comprehensive numeric distributions
print("--- Numerical Feature Statistics Summary ---")
print(df[['Quantity', 'Unit_Price', 'Shipping_Cost', 'Customer_Rating', 'Total_Sales']].describe().round(2))

# 2. Extract correlation insights
print("\n--- Feature Correlation Strengths Matrix ---")
numeric_df = df[['Quantity', 'Unit_Price', 'Shipping_Cost', 'Customer_Rating', 'Total_Sales']]
print(numeric_df.corr().round(3))


📊 TASK 4: DESCRIPTIVE STATISTICS & METRIC AUDITING

--- Numerical Feature Statistics Summary ---
       Quantity  Unit_Price  Shipping_Cost  Customer_Rating  Total_Sales
count   5000.00     5000.00        5000.00          4900.00      5000.00
mean       1.87      253.99          27.44             4.15       473.91
std        1.08      139.99          13.02             1.12       406.34
min        1.00       10.09           5.00             1.00        10.62
25%        1.00      132.69          16.14             4.00       194.32
50%        1.00      253.06          27.26             5.00       361.08
75%        2.00      371.42          38.78             5.00       612.00
max        5.00      499.95          49.99             5.00      2492.20

--- Feature Correlation Strengths Matrix ---
                 Quantity  Unit_Price  Shipping_Cost  Customer_Rating  \
Quantity            1.000      -0.005          0.008            0.010   
Unit_Price         -0.005       1.000          0.010  